# 3/2/26

## Worked on BIRD Bench evaluation scrip
Evaluation scrip did not work out of the box. Directions for location of files and directories between the README and evalution.sh script had conflicts. Evaluation scrip expected jsonl ground truth files but only json was given in the dataset.

Got too many errors from sqlite database. Moving on to MySQL. SQLite needs to split the databases up which narrows the scope. The goal of this project is to have a very large scope.

Reinstalled MySQL after loosing password.

Created a baseline py script for MySQL, tried to run it on the hpc but both h100 gpus were in partial use. Running gpt-oss-20b on the more available GPU at .80 gpu memeory would take 1.5 hours at least to generate 500 responses. This baseline only takes 1 LLM call, the furture versions are expected to take anywhere from 3-10. I am going to try again tomorrow with a smaller model.

It is becoming clear LLMs are more expensive to run than I expected. My expectations were built on results from research papers and my expereince at Samsung, both places where large GPUs and possibly multiple of them are dedicated to the project. I see Prof. Flores' interest in building a system on small models. 

## 3/3/26 Beginning Work with Medium-Small models --- Testing Phi-3 --- Enabling Debug in VLLM --- 

Bird has some models already evalauted as bench marks. I am starting with a few of these models to use as a reference for my own baseline. 

My first model I wanted to download was meta-llama/Llama-3-8B-Instruct. It requires approval on hugging face to download, is signed up but need to wait for approval.In the mean time it tried to downlaod it through ollama. Ollama, however, does not have a download that would easily accessible from the gpu node (considered trying to move the donwloaded ollama bnary to my shared space, but seems like too much of a hassle). Decided to move onto a differnt model while I wait for meta to approve my request.

To keep track of all models and make sure the models are not wiped from temp storage, i moved the hugging face home directory to `export HF_HOME=/home/012155624/hg_cache/`. I downloaded the model with `hf download microsoft/Phi-3-medium-128k-instruct     --include "*.safetensors" "*.json" "tokenizer*"`. I ran the model with vllm serve `/home/012155624/hg_cache/hub/models--microsoft--Phi-3-medium-128k-instruct/snapshots/a088b37c71d441ab6d862bb3fcfe6165b3014702 --host 0.0.0.0 --port 8000`.

Noteable about phi-3-medium-128k, 128k refers to the context length. The model is a 14B pararmeter model.

New issue, the phi-3-medium is not recieving all of the conversation history from langchain.

```
(APIServer pid=1737898) DEBUG 03-03 13:29:10 [entrypoints/logger.py:39] Request chatcmpl-bfdef6dc50df1064 details: prompt: "<|user|>\nWhat is the ratio of customers who pay in EUR against customers who pay in CZK? You may find this informaiton helpful: ratio of customers who pay in EUR against customers who pay in CZK = count(Currency = 'EUR') / count(Currency = 'CZK').<|end|>\n<|assistant|>\n", prompt_token_ids: [32010, 1724, 338, 278, 11959, 310, 20330, 1058, 5146, 297, 382, 4574, 2750, 20330, 1058, 5146, 297, 315, 29999, 29968, 29973, 887, 1122, 1284, 445, 1871, 1249, 265, 8444, 29901, 11959, 310, 20330, 1058, 5146, 297, 382, 4574, 2750, 20330, 1058, 5146, 297, 315, 29999, 29968, 353, 2302, 29898, 29907, 10880, 353, 525, 29923, 4574, 1495, 847, 2302, 29898, 29907, 10880, 353, 525, 29907, 29999, 29968, 2824, 32007, 32001], prompt_embeds shape: None.
```

Clearly the model is only recieving the user message when the code is meant to send the system message and state message (at this point a previous tool call) and tool messages:
```
llm.invoke(
            [
                SystemMessage(
                    content=system_message_content
                )
            ]
            + state["messages"]
        )
```

vLLM debug mode:
```
export VLLM_LOGGING_LEVEL=DEBUG
--enable-log-requests --enable-log-outputs
```

I did not get to testing meta-llama/Llama-3-8B-Instruct. I will try at a later time.

# 3/4/26 Fixing Evalaution Scripts --- Running Baseline with Llama 3 8B --- 
Evalaution scripts didnt work, I moved from trying to get the .sh script working on my windows machine to just setting the defualt values in the args to the values i need. This allowed for me to run the python files in VS debug mode. I discovered an error in evaluation_utils when connecting to the database that was being handled by a try catch. I changed a few values form unix socket address to port 3306.

Sidenote: I am working in windows because wsl is difficult and VMs run very slow on my local machine (laptop).

I achieved my first evalaution with gpt-oss-20b, a 43 soft-f1 score. Unfortunatly there is not gpt-oss in the BIRD read me bench marks to compare the quality of my benchmark. Next step is to run Meta-Llama-3-8B-Instruct.

Meta-Llama-3-8B-Instruct downloaded without the nessecary config.json file. I found the config file in the huggingface repo under files and versions. Copy and pasting the file into the snapshots folder solved the issue.

I had to remove the example values of each table to make the prompt fit. But I cant remember if I removed it before gpt oss or before llama 3

Successfully ran the llama-3 benchmark and eval, but results were very low.

###

## 3/5/26
Cleaned llama-3 benchmarks results because I thought the \n's were causing an error. It did not improve the f1 score.

Did some research on context length. If vllm recieves an input that is larger than the models context length (by defualt context length is founs in the model's config file) it takes a first in first out policy and will trunkate the beginning of the prompt. I guess the assumption is that the most recent information is the most relevant.

### Llama 3 context length matches tokenized length of db show tables
Additionally THIS MAY BE IMPORTANT FOR WRITING THESIS, the llama 3 8b's context size is 8k while the retreive schema, user message and system prompt in the basline averages out to 8k tokens. This would be a great setup for showing that using tools it helpful.

Started on a agent and LLM call to get_table_schema tool that will retrieve only the relevant table schema's

I discovered that OpenAiChat.bind_tools can take a structured response arg and that OpenAIChat.with_structured_output will take tools. Tools in structured output requires:
```
- `method` is `'json_schema'` (default).
            - `strict=True`
            - `include_raw=True`
```

Discovered and began to implement structured response in generation agent. This is going to be crucial in isolating the SQL for BIRD evaluation.

# 3/6/2026

### Reasoning and Reducing max_ouput
Qwen Models have reasoning turned on by default, the reasoning in this Had to reduce the max_output size of both llm to 2048 because the reasoning was taking too long. This reduced latency greatly. 

However the error still persisted. I turned thinking off by including /no_think at the end of the system prompt.

### Table Call failed
Question 78 seems to be getting the same problem. The schema retrieval model keeps trying to retrieve tables that do not exist.

```
User Question: "Write the full names of students who received funds on the date of 9/9/2019 and include the amount received. You may find this informaiton helpful: full name refers to first_name, last_name, amount of funds received refers to amount, received funds on date refers to date_received"

requested table_list:
  - students
  - funds_received
```

I have several options to solve this problem:
1. Make the get schema agent loop, giving it multiple attempts to retrieve the correct tables. However this would increase the LLM call count.
2. Increase the amount of information given to the LLM. I could give the model the column names to help it inform it's decisions.
3. Put a try except stament around the agent invoke command.

Solution 3 turned out to be the most effective. There were only 2 errors, both of them were LLM ouput exceeded parameter size. Qwen's chain of thought got cuaght in a loop of listing either database tables names (usually happned when the previos LLM didnt call any real tools) or listing possilble values in a column.

### LLM calling get schema
The model responisble for retriving the approprate column names was not super good at it's job (using qwen3-4B). It was not uncommon for the model to request fabricated tables names, usually inspired by the user question or provided evidence

# 3/9/2026 model size eval on V2.1  --- Agent Env Looping
Tried to upgrade to Qwen3.5 but vLLM doesnt seem to support it yet. Running Qwen-3 4B, 8B, 14B, 32B on V2.1 and Baseline.

Had to update the evaluation and baseline script.

Reran gpt-oss-20B on Baseline with a 2048 output limit, reduces latency from 1hour to 30 mins (need to double check the no max token run time) and didn't effect accuracy.

# 3/10/2026  GOAL: Run all qewn models, gpt-oss-20b, and another model family on V1Baseline and V2
Added a few lines of code to V2 to catch the tables the LLM with deciding to retrieve. The hope is to measure how accuratley they are identifying which tables are relavent.

I am starting to set up a multiple model pipeline, as I do I am realizing that to get multiple models on 1 gpu, you need to split the gpu utlization between the two vllm instances. I have a feeling this will drastically increase latency even in the smaller models.

I was wrong, I ran Qwen3-4B at .45 gpu utlization and it was able to go through the dataset in 10 mins. Now time to set it up with a differnt model for generation.

Set up two qwen3's 4B and 14B for a dual model test. First issue I ran into was having enough vram which was addressed above. Now I am running into     "151": "ERROR: Error code: 400 - {'error': {'message': \"'max_tokens' or 'max_completion_tokens' is too large: 1024. This model's maximum context length is 2048 tokens and your request has 1635 input tokens (1024 > 2048 - 1635). (parameter=max_tokens, value=1024)\", 'type': 'BadRequestError', 'param': 'max_tokens', 'code': 400}}", 

Around 350 there are several questions that can trigger the above error.

### Max tokens, vLLM and Langgraph
We need to reduce the max_model_len in vllm so that both model can fit on one gpu. Reducing the max model len in vllm args too much can cause an error were the combined input and output is too large. 

Langgraph parameter for model max_completion_tokens can set so that the response does not get too large, but the input varaies greatly based on which table is being retrieved. A max of 8192 in vllm for both models, 1024 for tool llm and 2038 for gen llm in langgraph seems to work.

With these varaibles and gpu usage set to .45 on both we are able to run two models with 75867MiB /  81559MiB of gpu memory used.


### End of Day --- Using two gpus.



# 3/11/2026 Pivot from smaller models to smaller max_model_len
Test this yestard lead me to believe that running two models (a smaller and a large) was overall worse. The effect on GPU usage was comparable only if both models were reduces to less gpu usage (~45% each) and smaller max_model_lens (8192 tokens). In these conditions the model had lower latency, but lower accuracy as well, most similar to the baseline results of the smaller model (qwen3-4B baseline ~21 F1, Qwen3-4B & 14B ~ TODO: ReRun These two models on one GPU). When splitting the models between two gpus, Qwen4B and Qwen14B achieved 24.29 Soft F1 score. Up from Qwen4B 20.95, but down from 14B 26.45 while using twice as many GPUs. Conclusion: using a smaller model decreases the performance, and achieving a signficant enough decrease inlatency would require two gpu's, counter to this project's objective of recourse usage low. 

I instead want to pivot to using one model with a decreased attention length, set by max_model_len in vLLM. My hypothesis is that by decreasing the max_model_len, we can decrease gpu useage and latency, and maintain accuracy. While many AI and agentic tasks involve summarizing or generating long texts, Text-to-SQL also does this but my goal is to reduce the input to the LLM to make it more accurate.

# End of the day
It didnt work. Turns out making more space for the kv-cache only allows for more parrallel requests. I could try out parellel generation... But that is something I will leave for tomorrow. For now I have no way to optimize the pipeline I have built. My results so far say that it is more effective to through the whole database at a big model than split tasks amoung smaller models. I need to make sure this is true by running large models through the baseline and measuring latency.

### TODO:
* Rerun latency tests on baseline and V3
* Really need to rerun accuracy too, rerun all tests
* Test if vllm can handle multiple calls at once with langgraph parallel workflow. Best chance I have of saving this thesis.
* I also want to set up parralizaiton in the for loop to speed up testing. Hopefully running 3 workers will reduce run time from 30 mins to 10.
* I want to combine the tables and token count result json files into one meta data file that will include latency. I will end up with 2 json files: 1 with resutls to be evaluated by BIRD, and 2 will hold meta data for the agent.



### ToDo
* ReRun 1GPU, but split gpu utilization evenly between small model and large model (maybe .3 & .7)
* Create one "true" agent that

# 3/14/2026
Rerunning evaluation of Sequential (v2) with threading with two models and again with one model.

Decided to test two models across to gpus because caculating the optimal amount of gpu usage for two models on one would be a hassle. I expect the latency of two gpus two models will be comparable (and slightly beter) to one gpu one model, while two gpus two model will be worse.

# 3/16/2026
Testing two models on one gpu. Figured out you can get the model memory requirements by adding of the size of the safetensors or pytorch_model.bin files. 

* Qwen3-14B ~30GB 
* Qwen3-8B 
* Qwen3-4B ~9GB

First test, running Qwen3 14B at .65 gpu usage and qwen3-4B at .25.

vllm serve /home/012155624/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c/ --host 0.0.0.0 --port 8000 --enable-log-requests --enable-log-outputs  --enable-auto-tool-choice --max-model-len 8192  --tool-call-parser hermes